In [0]:
import uuid
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F

In [0]:
def ingest_to_bronze(
    spark: SparkSession,
    source_name: str,
    source_dataset: str,
    landing_path: str,
    schema_location: str,
    checkpoint_location: str,
    target_table: str,
    file_format: str = "json",
    trigger_once: bool = True,
) -> dict:
    """
    Reads new files from a Volume landing path via Autoloader and appends
    them to a bronze Delta table, tagging each row with lineage metadata.

    One batch_id is generated per call (not per row) so Balance checks can
    group and count rows belonging to this specific run.
    """
    batch_id = str(uuid.uuid4())

    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(landing_path)
    )

    df_enriched: DataFrame = (
        df\
            .withColumn("_source_system", F.lit(source_name))
            .withColumn("_source_dataset", F.lit(source_dataset))
            .withColumn("_source_file", F.col("_metadata.file_path"))
            .withColumn("_file_size_bytes", F.col("_metadata.file_size"))
            .withColumn("_ingest_ts", F.current_timestamp())
            .withColumn("_batch_id", F.lit(batch_id))
    )

    query = (
        df_enriched.writeStream.format("delta")
        .option("checkpointLocation", checkpoint_location)
        .outputMode("append")
        .trigger(availableNow=True) if trigger_once else df_enriched.writeStream
    )

    stream_query = query.toTable(target_table)
    stream_query.awaitTermination()

    progress = stream_query.recentProgress
    rows_written = sum(
        source.get("numInputRows", 0)
        for p in progress
        for source in p.get("sources", [])
    )

    files_processed = sum(
        int(source.get("metrics", {}).get("numFilesProcessed", 0))
        for p in progress
        for source in p.get("sources", [])
    )

    return {
        "source_system": source_name,
        "source_dataset": source_dataset,
        "target_table": target_table,
        "batch_id": batch_id,
        "rows_written": rows_written,
        "files_processed": files_processed,
        "batches": len(progress),
    }